# Task Coding Demo — The Scale of Animal Agriculture

**Task coding** means running code one cell at a time, seeing the results at each step before moving on.
There's no "Run All" button mentality here — you execute, inspect, understand, then continue.

In this notebook, we'll use code to explore real data about global animal agriculture — the system we're building tools to change.
Each cell builds on the last, and by the end you'll have gone from raw data to visual insight, one step at a time.

**Before you start: File → Save a copy in Drive**

## Step 1: Load the Data

We're pulling real FAO data on animals slaughtered globally, published by [Our World in Data](https://ourworldindata.org/).
This dataset covers every country from 1961 to the present, broken down by species.

We'll fetch it directly from GitHub so there's nothing to download.

In [ ]:
import requests
import csv
from io import StringIO

url = "https://raw.githubusercontent.com/owid/owid-datasets/master/datasets/Animals%20slaughtered%20for%20meat%20-%20FAO%20(2023)/Animals%20slaughtered%20for%20meat%20-%20FAO%20(2023).csv"

try:
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    reader = csv.DictReader(StringIO(response.text))
    data = list(reader)
    columns = list(data[0].keys())
    print("Columns:", columns)
    print()
    for row in data[:5]:
        print(row)
    print(f"\nTotal rows: {len(data):,}")
    print("\u2705 Data loaded from Our World in Data.")
except Exception as e:
    print(f"Could not fetch remote data: {e}")
    print("Loading hardcoded fallback dataset...")
    columns = ["Entity", "Year",
               "Livestock primary - Meat, chicken - 1058 - Producing Animals/Slaughtered - 5510 - Head",
               "Livestock primary - Meat, pig - 1035 - Producing Animals/Slaughtered - 5510 - Head",
               "Livestock primary - Meat, cattle - 867 - Producing Animals/Slaughtered - 5510 - Head",
               "Livestock primary - Meat, sheep - 977 - Producing Animals/Slaughtered - 5510 - Head",
               "Livestock primary - Meat, goat - 1017 - Producing Animals/Slaughtered - 5510 - Head",
               "Livestock primary - Meat, duck - 1069 - Producing Animals/Slaughtered - 5510 - Head",
               "Livestock primary - Meat, turkey - 1080 - Producing Animals/Slaughtered - 5510 - Head"]
    data = [
        {"Entity": "World", "Year": "2021",
         columns[2]: "72852879000", columns[3]: "1382457000",
         columns[4]: "302168000", columns[5]: "620247000",
         columns[6]: "497560000", columns[7]: "3658000000",
         columns[8]: "656396000"},
        {"Entity": "World", "Year": "2020",
         columns[2]: "72010342000", columns[3]: "1375320000",
         columns[4]: "299830000", columns[5]: "615880000",
         columns[6]: "491230000", columns[7]: "3590000000",
         columns[8]: "641000000"},
        {"Entity": "World", "Year": "2000",
         columns[2]: "40404680000", columns[3]: "1235370000",
         columns[4]: "290530000", columns[5]: "541360000",
         columns[6]: "364670000", columns[7]: "2100000000",
         columns[8]: "570000000"},
        {"Entity": "World", "Year": "1980",
         columns[2]: "15924620000", columns[3]: "825750000",
         columns[4]: "240600000", columns[5]: "434200000",
         columns[6]: "244890000", columns[7]: "800000000",
         columns[8]: "250000000"},
        {"Entity": "World", "Year": "1961",
         columns[2]: "5765970000", columns[3]: "375400000",
         columns[4]: "175210000", columns[5]: "316100000",
         columns[6]: "159000000", columns[7]: "300000000",
         columns[8]: "100000000"},
        {"Entity": "United States", "Year": "2021",
         columns[2]: "9680000000", columns[3]: "131563000",
         columns[4]: "33742000", columns[5]: "2225000",
         columns[6]: "800000", columns[7]: "330000000",
         columns[8]: "223000000"},
        {"Entity": "India", "Year": "2021",
         columns[2]: "3580000000", columns[3]: "9500000",
         columns[4]: "35300000", columns[5]: "83000000",
         columns[6]: "72000000", columns[7]: "250000000",
         columns[8]: "5000000"},
        {"Entity": "Brazil", "Year": "2021",
         columns[2]: "6200000000", columns[3]: "45600000",
         columns[4]: "34500000", columns[5]: "11400000",
         columns[6]: "5700000", columns[7]: "50000000",
         columns[8]: "25000000"}
    ]
    print("Columns:", columns)
    print()
    for row in data[:5]:
        print(row)
    print(f"\nTotal rows: {len(data):,} (fallback dataset)")
    print("\u2705 Fallback data loaded.")

## Step 2: Find the Global Numbers

The dataset has rows for individual countries *and* a "World" aggregate.
Let's filter for the world totals in the most recent year and see what the numbers actually look like.

In [ ]:
try:
    # Filter for world data and find the most recent year
    world_rows = [r for r in data if r["Entity"] == "World"]
    latest_year = max(int(r["Year"]) for r in world_rows)
    latest = [r for r in world_rows if int(r["Year"]) == latest_year][0]

    # Build a clean species → count mapping
    species_map = {}
    for col in columns:
        if "Slaughtered" in col:
            # Extract species name from column header
            name = col.split("Meat, ")[1].split(" -")[0].capitalize()
            val = latest.get(col, "")
            if val:
                species_map[name] = int(float(val))

    print(f"Global animals slaughtered — {latest_year}")
    print("=" * 45)
    total = 0
    for species, count in sorted(species_map.items(), key=lambda x: -x[1]):
        print(f"  {species:<12} {count:>20,}")
        total += count
    print("=" * 45)
    print(f"  {'TOTAL':<12} {total:>20,}")
    per_second = total / (365.25 * 24 * 3600)
    print(f"\nThat's roughly {per_second:,.0f} animals per second.")
    print("\u2705 Summary complete.")
except Exception as e:
    print(f"Error building summary: {e}")
    print("Make sure you ran the data-loading cell above first.")

> 💡 **Pause and think about that number.** This is why we're here. Every tool you build in this course is a step toward changing this system.

## Step 3: Visualise the Trend

Numbers in a table are useful. A chart makes the trajectory *visceral*.
Let's plot chicken slaughter over time — it's the largest number and the most dramatic growth curve.

In [ ]:
import matplotlib.pyplot as plt

try:
    # Find the chicken column
    chicken_col = [c for c in columns if "chicken" in c.lower()][0]

    # Extract world chicken data over time
    world_rows = [r for r in data if r["Entity"] == "World"]
    years = sorted([int(r["Year"]) for r in world_rows])
    counts = []
    for y in years:
        row = [r for r in world_rows if int(r["Year"]) == y][0]
        val = row.get(chicken_col, "0")
        counts.append(int(float(val)) / 1e9 if val else 0)

    plt.figure(figsize=(10, 5))
    plt.plot(years, counts, color="#c0392b", linewidth=2)
    plt.title("Global Chicken Slaughter (1961\u2013Present)", fontsize=14)
    plt.xlabel("Year")
    plt.ylabel("Billions of animals")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\u2705 Chart rendered.")
except Exception as e:
    print(f"Error creating chart: {e}")
    print("Make sure you ran the data-loading cell first.")

In [ ]:
try:
    # Define the species we want to stack
    target_species = ["chicken", "pig", "cattle", "sheep", "goat"]
    species_cols = {}
    for sp in target_species:
        matches = [c for c in columns if sp in c.lower() and "Slaughtered" in c]
        if matches:
            species_cols[sp.capitalize()] = matches[0]

    world_rows = [r for r in data if r["Entity"] == "World"]
    years = sorted(set(int(r["Year"]) for r in world_rows))
    row_by_year = {int(r["Year"]): r for r in world_rows}

    # Build series for each species
    series = {}
    for label, col in species_cols.items():
        series[label] = [int(float(row_by_year[y].get(col, "0") or "0")) / 1e9 for y in years]

    colors = ["#e74c3c", "#e67e22", "#2980b9", "#27ae60", "#8e44ad"]
    plt.figure(figsize=(10, 5))
    plt.stackplot(years, *series.values(), labels=series.keys(), colors=colors, alpha=0.85)
    plt.title("Animals Slaughtered by Species \u2014 Global", fontsize=14)
    plt.xlabel("Year")
    plt.ylabel("Billions of animals")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\u2705 Stacked chart rendered.")
except Exception as e:
    print(f"Error creating stacked chart: {e}")
    print("Make sure you ran the data-loading cell first.")

> 💡 **Key Concept:** See how each cell built on the last? We loaded data, explored it, extracted insights, and visualised trends — all step by step, with visible results at every stage. That's task coding. Now imagine doing this with social media sentiment data, policy documents, or supply chain information. That's what advocacy technology looks like.

> 🎯 **Your Turn:** Filter the data for a specific country you're interested in. What does animal agriculture look like there? How has it changed over time?

In [ ]:
# Change this to any country in the dataset
my_country = "..."  # e.g. "Brazil", "India", "United States", "Australia"

try:
    country_rows = [r for r in data if r["Entity"] == my_country]
    if not country_rows:
        print(f"No data found for '{my_country}'. Check spelling — it must match exactly.")
        print("Some available entities:")
        entities = sorted(set(r["Entity"] for r in data))
        for e in entities[:20]:
            print(f"  {e}")
    else:
        chicken_col = [c for c in columns if "chicken" in c.lower()][0]
        years = sorted(set(int(r["Year"]) for r in country_rows))
        row_by_year = {int(r["Year"]): r for r in country_rows}
        counts = [int(float(row_by_year[y].get(chicken_col, "0") or "0")) / 1e9 for y in years]

        plt.figure(figsize=(10, 5))
        plt.plot(years, counts, color="#c0392b", linewidth=2)
        plt.title(f"Chicken Slaughter \u2014 {my_country}", fontsize=14)
        plt.xlabel("Year")
        plt.ylabel("Billions of animals")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f"\u2705 Chart for {my_country} rendered.")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you ran the data-loading cell and set my_country above.")